# Day 7 — Pandas: Multi-Agg Groupby, Pivot Tables, Strings, Datetime

## Concept 1: `.groupby()` with multiple aggregations (`.agg()`)

So far you've done things like `df.groupby("club")["attendance"].mean()` — one column, one function. `.agg()` lets you run **several functions at once**, and/or on **several columns at once**.

```python
df.groupby("club")["attendance"].agg(["mean", "max", "count"])
```

This gives you a small table: for each club, the mean attendance, the max attendance, and how many rows (students) were in that group — all in one call, instead of three separate `.groupby()` calls.

You can also aggregate multiple columns differently using a dictionary:

```python
df.groupby("club").agg({
    "attendance": "mean",
    "score": "max"
})
```

This says: for `attendance`, give me the mean; for `score`, give me the max — different functions per column.

In [3]:
import pandas as pd
import numpy as np

## Exercise 1: Multi-agg groupby

Using `class_data` below, group by `"class"` and return the **mean**, **max**, and **min** of `"score"` for each class, all in one call.

In [5]:
class_data = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
    "class": ["A", "A", "B", "B", "A", "B"],
    "score": [85, 90, 78, 60, 95, 70]
})

In [6]:
def class_score_summary(df):
    """Group by 'class' and return mean, max, and min of 'score' for each class,
    all in a single .agg() call."""

    
    return df.groupby("class")["score"].agg(["mean", "max", "min"])

In [7]:
# Test cell
print(class_score_summary(class_data))

            mean  max  min
class                     
A      90.000000   95   85
B      69.333333   78   60


## Concept 2: Pivot tables (`.pivot_table()`)

A pivot table reshapes data into a spreadsheet-like summary — one column's values become new columns, another column's values become rows, and a third column gets summarized (usually with `mean`, `sum`, etc.) in the cells where they cross.

```python
df.pivot_table(values="score", index="class", columns="term", aggfunc="mean")
```

- `values` — the column being summarized
- `index` — becomes the rows of the result
- `columns` — becomes the columns of the result
- `aggfunc` — how to summarize when multiple rows land in the same cell (default is `"mean"`)

It's conceptually similar to `.groupby()` with two keys, just displayed as a grid instead of a long list — this is exactly what a pivot table in Excel does too.

## Exercise 2: Pivot table

Using `term_data` below, create a pivot table showing the **average score**, with `"class"` as rows and `"term"` as columns.

In [18]:
term_data = pd.DataFrame({
    "name": ["Alice", "Alice", "Bob", "Bob", "Carol", "Carol"],
    "class": ["A", "A", "A", "A", "B", "B"],
    "term": ["Term1", "Term2", "Term1", "Term2", "Term1", "Term2"],
    "score": [80, 85, 70, 75, 90, 95]
})

In [21]:
def score_pivot(df):
    """Return a pivot table of average score, with 'class' as rows
    and 'term' as columns."""
    return df.pivot_table( index = "class", columns = "term", values = "score", aggfunc ="mean") 
    

In [22]:
# Test cell
print(score_pivot(term_data))

term   Term1  Term2
class              
A       75.0   80.0
B       90.0   95.0


## Concept 3: String methods (`.str` accessor)

Real text data is messy — inconsistent capitalization, extra spaces, typos. pandas gives you the `.str` accessor to run string operations across an entire column at once (similar idea to `.dt` for dates, which you'll see next).

```python
df["name"].str.lower()               # lowercase every value
df["name"].str.contains("a")         # True/False per row, does it contain 'a'?
df["name"].str.replace("o", "0")     # replace characters
df["name"].str.strip()               # remove leading/trailing whitespace
```

These work just like normal Python string methods (`.lower()`, `.replace()`, etc.) but applied to *every row in the column at once*, instead of writing a loop.

## Exercise 3: Clean and filter text

Using `messy_names` below:
1. Strip extra whitespace from `"name"`
2. Make it lowercase
3. Return only the rows where the cleaned name contains the letter `"a"`

In [23]:
messy_names = pd.DataFrame({
    "name": ["  Alice", "BOB ", " carol ", "Dave", "eve "]
})

In [26]:
def clean_and_filter_names(df):
    """Strip whitespace and lowercase the 'name' column, then return
    only rows where the cleaned name contains 'a'."""
    st_df = df["name"].str.strip()
    return df[st_df.str.contains("a")]

In [27]:
# Test cell
print(clean_and_filter_names(messy_names))

      name
2   carol 
3     Dave


## Concept 4: Datetime handling

Dates often arrive as plain text (`"2024-01-15"`), which pandas won't understand as an actual date unless you convert it:

```python
df["date"] = pd.to_datetime(df["date"])
```

Once converted, the `.dt` accessor lets you pull out parts of the date, similar to how `.str` works for text:

```python
df["date"].dt.year
df["date"].dt.month
df["date"].dt.day_name()   # e.g. 'Monday'
```

This is especially useful once you start asking time-based questions, like "what's the average score by month" — you'd first extract the month with `.dt.month`, then `.groupby()` on it.

## Exercise 4: Mixed review — datetime + groupby

Using `submissions` below:
1. Convert `"submitted_at"` to an actual datetime column
2. Add a new column `"month"` with the month number extracted from it
3. Group by `"month"` and return the average `"score"` per month

In [28]:
submissions = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave"],
    "submitted_at": ["2024-01-15", "2024-01-20", "2024-02-10", "2024-02-25"],
    "score": [85, 90, 78, 60]
})

In [33]:
def avg_score_by_month(df):
    """Convert 'submitted_at' to datetime, extract the month into a new
    'month' column, then return the average score grouped by month."""
    df["submitted_at"] = pd.to_datetime(df["submitted_at"]) 
    month = df["submitted_at"].dt.month
    average = df.groupby(month)["score"].mean() 
    return average 

In [34]:
# Test cell
print(avg_score_by_month(submissions))

submitted_at
1    87.5
2    69.0
Name: score, dtype: float64
